# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip -q install duckdb huggingface_hub

In [5]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [6]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis
One row represents one content page during the selected analysis month (2026-03). Each row contains aggregated search performance, engagement, and content features used to estimate the page's refresh opportunity score.

### Time Window
This notebook uses the mid-panel month (2026-03). A middle month is chosen instead of the final month to avoid data leakage and to keep the last month for evaluation.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature Fields
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days

### Label (Proxy)
Content Opportunity Score (proxy), used to rank pages for refresh priority.

### Context Fields
- client_hash_id
- content_hash_id
- report_date

### Excluded Fields
- Future performance columns
- Label-derived columns
- Final month (2026-06)

These are excluded to prevent data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    COUNT(DISTINCT report_date) AS unique_dates
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content,unique_dates
0,9841378,331437,31


### Query 1 – Verify the Grain

This query verifies the grain of the selected data slice for March 2026.

**Observation:**

The results show the total number of daily records, the number of unique content pages, and the number of unique reporting dates. This confirms that the `fact_daily` table stores daily observations, and multiple daily records exist for each content page during the selected month. These daily records will later be aggregated to create one feature row per content page.

In [9]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### Query 2 – Verify the Time Window

This query verifies the number of records and the date range for the selected analysis month.

**Observation:**

The selected data slice contains **9,841,378** records. The data starts on **2026-03-01** and ends on **2026-03-31**, confirming that the analysis uses the complete month of March 2026 as defined in the data contract.

In [10]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND client_has_gsc IS TRUE
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


### Query 3 – Verify Data Availability

This query checks how many rows have Google Search Console data available for the selected month using the required `IS TRUE` filter.

**Observation:**

The query returned **3,611,061** rows where both `gsc_data_available` and `client_has_gsc` are **TRUE**. This confirms that the analysis uses only records with available Google Search Console data, as required by the data contract.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits

This dataset has several important limitations:

- Client history is unbalanced because different clients started collecting data at different times.
- Some clients have only Google Search Console (GSC) data, while others also have Google Analytics 4 (GA4) data.
- The Content Opportunity Score is a proxy label and not a real business-provided target.
- The dataset cannot directly measure content quality, user intent, or business impact.
- The results should be used as decision-support rather than absolute truth.

## 5. Five Feature Frame

In [11]:
features = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS impressions_90d,
    SUM(gsc_clicks) AS clicks_90d,
    AVG(gsc_avg_position) AS avg_position,
    SUM(ga4_sessions) AS sessions_90d,
    SUM(scroll_events) AS scroll_events_90d
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY content_hash_id
LIMIT 5
""").df()

features

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions_90d,clicks_90d,avg_position,sessions_90d,scroll_events_90d
0,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0
1,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0
2,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0
3,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0
4,content_1855a661b4d36130,429.0,1.0,4.209227,2.0,1.0


### Selected Features

| Feature | Available at decision time because... |
|---------|----------------------------------------|
| impressions_90d | It is calculated from historical search performance before deciding whether to refresh a page. |
| clicks_90d | Click history is already known before the refresh decision is made. |
| avg_position | Average search position is observed before the prediction period. |
| sessions_90d | Historical user sessions are available before making the decision. |
| scroll_events_90d | User engagement has already occurred and is known at decision time. |

## 6. Feature Leakage Experiment

### Deliberate Leakage Example

If we include a feature that is directly derived from the target (for example, the final Opportunity Score or a future performance value), the model may appear to perform almost perfectly because it already contains information about the correct answer.

This is called **data leakage**.

For the final model, all label-derived and future information must be removed so that predictions use only information available at the decision time.

In [15]:
features = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    SUM(ga4_sessions) AS sessions,
    SUM(scroll_events) AS scroll_events
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY content_hash_id
LIMIT 5000
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,avg_position,sessions,scroll_events
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0
1,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0
2,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0
3,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,1.0
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0


In [16]:
features["opportunity_score"] = (
    features["impressions"] * 0.4 +
    features["clicks"] * 0.3 +
    features["sessions"] * 0.2 +
    features["scroll_events"] * 0.1
)

features.head()

,content_hash_id,impressions,clicks,avg_position,sessions,scroll_events,opportunity_score
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0,456.6
1,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0,22.8
2,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0,60.4
3,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,1.0,572.1
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0,1113.4


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X = features[
    [
        "impressions",
        "clicks",
        "avg_position",
        "sessions",
        "scroll_events"
    ]
]

y = features["opportunity_score"]


features["opportunity_score"].isna().sum()

np.int64(20)

In [19]:
features_clean = features.dropna(
    subset=[
        "impressions",
        "clicks",
        "avg_position",
        "sessions",
        "scroll_events",
        "opportunity_score"
    ]
)

print("Before:", len(features))
print("After:", len(features_clean))

features_clean.head()


Before: 5000
After: 4980


,content_hash_id,impressions,clicks,avg_position,sessions,scroll_events,opportunity_score
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0,456.6
1,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0,22.8
2,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0,60.4
3,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,1.0,572.1
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0,1113.4


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


X = features_clean[
    [
        "impressions",
        "clicks",
        "avg_position",
        "sessions",
        "scroll_events"
    ]
]

y = features_clean["opportunity_score"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


model = RandomForestRegressor(
    random_state=42
)

model.fit(X_train, y_train)


pred = model.predict(X_test)


print("Honest R² Score:", r2_score(y_test, pred))

Honest R² Score: 0.9998650489110184


In [21]:
features_clean["leak_feature"] = features_clean["opportunity_score"]

features_clean.head()

/tmp/ipykernel_3237/2703726114.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features_clean["leak_feature"] = features_clean["opportunity_score"]


,content_hash_id,impressions,clicks,avg_position,sessions,scroll_events,opportunity_score,leak_feature
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0,456.6,456.6
1,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0,22.8,22.8
2,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0,60.4,60.4
3,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,1.0,572.1,572.1
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0,1113.4,1113.4


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X_leak = features_clean[
    [
        "impressions",
        "clicks",
        "avg_position",
        "sessions",
        "scroll_events",
        "leak_feature"
    ]
]

y = features_clean["opportunity_score"]


X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)


leak_model = RandomForestRegressor(
    random_state=42
)

leak_model.fit(X_train, y_train)

pred = leak_model.predict(X_test)


print("Leaky R² Score:", r2_score(y_test, pred))


Leaky R² Score: 0.9998991008358634


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.